<a href="https://colab.research.google.com/github/yy726/tool_use_demo/blob/main/notebooks/02_grpo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - GRPO on a mock tool environment

Continues from the SFT adapter of `01_sft.ipynb`. The planner now learns from its own rollouts: for each question it
runs up to three rounds of real tool calls against a local environment, and a rule-based reward scores the outcome.
Training uses TRL's `GRPOTrainer` with native multi-turn tool calling, as in Unsloth's *Qwen3 (4B) GRPO Tool Use*
notebook: tool results are masked out of the loss, and a tool error is sent back to the model instead of crashing.

In order:

1. **BFCL** on the SFT checkpoint.
2. Load the model with Unsloth's vLLM engine and the SFT adapter.
3. **GRPO**, checkpointing to Google Drive.
4. **PlannerQA-Bench** on the GRPO checkpoint; push adapter, merged weights and GGUF.
5. **BFCL** on the GRPO checkpoint, and a final base / SFT / GRPO comparison.

Use the same configuration as in `01_sft.ipynb` (same `SMOKE`, `MODEL_KEY`), so both notebooks share one Drive folder.
After a disconnect: *Run all* again.

## 1. Configuration

In [1]:
SMOKE = True                 # must match 01_sft.ipynb
MODEL_KEY = "qwen3-1.7b"     # must match 01_sft.ipynb
ENABLE_THINKING = False
REWARD_MODE = "graded_cost"  # "graded_cost" (default) | "graded" | "binary"
USE_DRIVE = True
RESUME = True
RUN_BFCL = True
WANDB_PROJECT = "tool-use-planner"
PUSH_TO_HUB = True
HF_PRIVATE = False
PUSH_GGUF = True
OVERRIDES = {}               # e.g. {"grpo_steps": 200}; see planner/config.py

## 2. Install and set up

In [2]:
%%capture
import os
import importlib.metadata as _md
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    def _pin(_p):
        try: return f'{_p}=={_md.version(_p).split("+")[0]}'
        except Exception: return _p
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    if is_t4:
        # Keep the machine's torch: upgrading it can pull a CUDA build the driver cannot load
        _pins = " ".join(_pin(_p) for _p in ("numpy", "pillow", "torch", "torchvision"))
        !uv pip install -qqq {_pins} bitsandbytes xformers unsloth
    else:
        _cmd = "uv pip install -qqq --upgrade vllm==0.19.1 {} {} torchvision bitsandbytes xformers unsloth"
        if os.system(_cmd.format(_pin("numpy"), _pin("pillow"))) != 0:
            os.system(_cmd.format("numpy", "pillow"))
    !uv pip install -qqq triton
    try: _torch_v = tuple(int(_p) for _p in _md.version("torch").split("+")[0].split(".")[:2])
    except Exception: _torch_v = ()
    _torchao = "torchao>=0.16.0" if _torch_v >= (2, 10) else "torchao>=0.16.0,<0.18.0"
    !uv pip install -qqq --no-deps --upgrade "{_torchao}"
!uv pip install transformers==5.15.1 "datasets>=4.7.0,<5.0.0"
!uv pip install --no-deps trl==1.13.0
!uv pip install -qqq --no-deps --force-reinstall "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo" "unsloth @ git+https://github.com/unslothai/unsloth"
!uv pip install -qqq bm25s==0.3.16 wandb

In [3]:
import importlib.metadata as _md, sys

def _installed(_p):
    try: return _md.version(_p)
    except Exception: return None

print(" | ".join(f"{_p} {_installed(_p) or 'NOT INSTALLED'}" for _p in ("torch", "numpy", "transformers", "trl", "unsloth", "vllm", "bitsandbytes")))
_missing = [_p for _p in ("torch", "transformers", "trl", "unsloth", "unsloth_zoo", "bm25s") if not _installed(_p)]
assert not _missing, f"The install cell failed for {_missing}. Remove its first line (%%capture), run it again and read the error."
_stale = [_m for _m, _p in (("numpy", "numpy"), ("PIL", "pillow"), ("torch", "torch"))
          if _m in sys.modules and getattr(sys.modules[_m], "__version__", "").split("+")[0] != (_installed(_p) or "").split("+")[0]]
assert not _stale, f"The install replaced {_stale} after this session had loaded it. Choose Runtime > Restart session, then Run all again."

torch 2.10.0 | numpy 2.2.6 | transformers 5.15.1 | trl 1.13.0 | unsloth 2026.11.4 | vllm 0.19.1 | bitsandbytes 0.50.2


In [4]:
import os, subprocess, sys

REPO_URL = "https://github.com/yy726/tool_use_demo.git"
REPO_BRANCH = "main"

if os.path.isdir("planner"):                      # already inside a checkout
    REPO_DIR = os.getcwd()
elif os.path.isdir("../planner"):
    REPO_DIR = os.path.abspath("..")
else:
    REPO_DIR = "/content/tool_use_demo"
    if not os.path.isdir(REPO_DIR):
        url = REPO_URL
        try:                                      # private repo: add a GITHUB_TOKEN secret
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
            if token: url = url.replace("https://", f"https://{token}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, url, REPO_DIR], check=True)
    else:                                         # an older clone: move it to the latest commit of REPO_BRANCH
        for cmd in (["fetch", "--depth", "1", "origin", REPO_BRANCH], ["checkout", "-f", "-B", REPO_BRANCH, "FETCH_HEAD"]):
            done = subprocess.run(["git", "-C", REPO_DIR] + cmd, capture_output=True, text=True)
            if done.returncode: print("WARNING: git", cmd[0], "failed, the code in", REPO_DIR, "may be stale:", done.stderr.strip())
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
head = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("repo:", REPO_DIR, "@", head)

repo: /content/tool_use_demo @ d0f145c Fix 01_sft GPU OOM (bm25s started JAX), silent install failure; add Colab badges


In [5]:
OVERRIDES = dict(OVERRIDES, reward_mode=REWARD_MODE)
import unsloth  # import before transformers / trl so its patches apply
import json, random
from planner.config import get_config
from planner.persist import StageCache, persist_dir, read_secrets
from planner.setup_env import prepare

cfg = get_config("smoke" if SMOKE else "a100", MODEL_KEY, enable_thinking=ENABLE_THINKING, **OVERRIDES)

print("secrets:", read_secrets())                 # HF_TOKEN, WANDB_API_KEY, GITHUB_TOKEN from Colab Secrets
HF_USER = None
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login, whoami
    login(token=os.environ["HF_TOKEN"])
    HF_USER = whoami()["name"]

PERSIST = persist_dir(cfg.exp_name, USE_DRIVE)    # Google Drive: survives a disconnect
LOCAL = f"/content/work/{cfg.exp_name}"            # local disk: large merged weights
os.makedirs(LOCAL, exist_ok=True)
CACHE = StageCache(PERSIST, enabled=RESUME)
CACHE.check_config(cfg.fingerprint())
print("persist dir:", PERSIST, "| finished stages:", CACHE.done())

env = prepare(cfg, os.path.join(PERSIST, "env"))  # world + knowledge base + sft / grpo / eval tasks

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
/usr/local/lib/python3.13/dist-packages/unsloth/_gpu_init.py:490: UserWarning: Unsloth: torchcodec is installed but cannot load its native libraries although FFmpeg is on the loader path; likely an FFmpeg major it does not support (it takes 4 to 8), a missing CUDA NPP runtime (nvidia-npp), or a build that does not match this torch; audio datasets decode through soundfile and PyAV instead (wav/flac/mp3/ogg, m4a/aac/webm). torchcodec 0.17.0+cu130 is incompatible with torch 2.10.0+cu128; install a matching build with `pip install --index-url https://download.pytorch.org/whl/cu128 'torchcodec>=0.10,<0.11.0'`.
  disable_torchcodec_if_broken()


Unsloth: xformers is installed but cannot load with torch 2.10.0+cu128, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
secrets: {'HF_TOKEN': True, 'WANDB_API_KEY': False, 'GITHUB_TOKEN': False}
Mounted at /content/drive
persist dir: /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke | finished stages: ['bench_base', 'bench_sft', 'bfcl_base']
Loaded cached environment from /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/env
Knowledge base: 8794 passages
  sft      96 tasks  {'kb_bridge': 12, 'kb_compare': 12, 'weather_lookup': 12, 'election_lookup': 12, 'fanout_calc': 12, 'tool_bridge': 12, 'composite': 12, 'no_tool': 12}
  grpo    128 tasks  {'kb_bridge': 16, 'kb_compare': 16, 'weather_lookup': 16, 'election_lookup': 16, 'fanout_calc': 16, 'tool_br

## 3. BFCL on the SFT checkpoint

Run now, while the GPU is free. Base-model scores come from the cache written by `01_sft.ipynb`.

In [6]:
from planner import bfcl
from planner.unsloth_utils import resolve_merged

SFT_LORA = os.path.join(PERSIST, "sft_lora")
SFT_MERGED = os.path.join(LOCAL, "sft_merged")
STAGE = f"grpo_{cfg.reward_mode}"                  # reward ablations get their own folders and cache entries
GRPO_DIR = os.path.join(PERSIST, STAGE)
GRPO_LORA = os.path.join(PERSIST, f"{STAGE}_lora")
GRPO_MERGED = os.path.join(LOCAL, f"{STAGE}_merged")
HUB_STAGE = "grpo" if cfg.reward_mode == "graded_cost" else STAGE.replace("_", "-")
assert os.path.exists(os.path.join(SFT_LORA, "adapter_model.safetensors")), "Run 01_sft.ipynb first (same SMOKE / MODEL_KEY)."

BFCL = {"base": bfcl.run_stage(CACHE, "base", cfg, PERSIST, enabled=RUN_BFCL)}
merged = resolve_merged(cfg, "sft", SFT_MERGED, HF_USER) if RUN_BFCL and not CACHE.has("bfcl_sft") else None
BFCL["sft"] = bfcl.run_stage(CACHE, "sft", cfg, PERSIST, local_model_path=merged, enabled=RUN_BFCL)
# A GRPO checkpoint that is finished but not scored yet (see the last section) is scored now, while the GPU is free.
if CACHE.has(f"bfcl_{STAGE}"):
    BFCL["grpo"] = CACHE.load(f"bfcl_{STAGE}")
elif RUN_BFCL and os.path.exists(os.path.join(GRPO_LORA, "adapter_model.safetensors")):
    BFCL["grpo"] = bfcl.run_stage(CACHE, STAGE, cfg, PERSIST, local_model_path=resolve_merged(cfg, HUB_STAGE, GRPO_MERGED, HF_USER), enabled=True)
shown = {k: v for k, v in BFCL.items() if v}
if shown: print(bfcl.format_table(shown))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

$ /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model /content/work/planner_qwen3-1.7b_smoke/sft_merged --port 1053 --dtype bfloat16 --gpu-memory-utilization 0.85 --trust-remote-code > /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/bfcl/sft/vllm_server.log
$ /content/bfcl_venv/bin/python -m planner.bfcl generate --model planner/qwen3-nothink-FC --test-category simple_python,parallel,irrelevance --backend vllm --skip-server-setup --num-threads 64 --result-dir /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/bfcl/sft/result --local-model-path /content/work/planner_qwen3-1.7b_smoke/sft_merged
$ /content/bfcl_venv/bin/python -m planner.bfcl evaluate --model planner/qwen3-nothink-FC --test-category simple_python,parallel,irrelevance --result-dir /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/bfcl/sft/result --score-dir /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/bfcl/sft/score
| BFCL (official checker) | base | sft |
|-

## 4. Load the model and the SFT adapter

`fast_inference=True` gives Unsloth's vLLM engine, which generates the rollouts with the LoRA being trained.

In [7]:
from unsloth import FastLanguageModel
from planner.bench import format_report, run_benchmark, text_backend
from planner.chat import setup_chat_format
from planner.tasks import build_prompt
from planner.unsloth_utils import gpu_report, has_vllm, hf_generate_fn, load_lora_weights, load_model, vllm_generate_fn

MAX_NEW = cfg.bench_max_new_tokens * (4 if cfg.enable_thinking else 1)

def bench(name, lora_dir=None):
    """PlannerQA-Bench for the current weights (or a saved LoRA when the vLLM engine is on). Cached."""
    def run():
        if FAST:
            generate = vllm_generate_fn(model, tokenizer, lora_dir, MAX_NEW)
        else:
            FastLanguageModel.for_inference(model)
            generate = hf_generate_fn(model, tokenizer, MAX_NEW, cfg.bench_batch)
        return run_benchmark(env.tasks["eval"], text_backend(fmt.render_prompt, generate), batch_size=cfg.bench_batch, progress=True)
    return CACHE.get_or_run(f"bench_{name}", run)

FAST = has_vllm()                                  # no vLLM on a T4: rollouts fall back to transformers (slow)
model, tokenizer = load_model(cfg, fast_inference=FAST, gpu_memory_utilization=0.6)
fmt = setup_chat_format(tokenizer, enable_thinking=cfg.enable_thinking)
load_lora_weights(model, SFT_LORA)

BENCH = {"base": CACHE.load("bench_base") if CACHE.has("bench_base") else None, "sft": bench("sft", SFT_LORA)}
BENCH = {k: v for k, v in BENCH.items() if v}
print(format_report(BENCH))

INFO 11-13 18:12:48 [vllm_utils.py:1161] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.11.4: Fast Qwen3 patching. Transformers: 5.15.1. vLLM: 0.19.1.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not planning a device map; vLLM places its own weights. Using `sequential`.
Unsloth: Standby mode is enabled. Changing `gpu_memory_utilization` to 0.87875.
Unsloth: Patched vLLM's piecewise graph-partition pass so compilation_config=3 (CUDA graphs) works with LoRA.
Unsloth: Not an error, but `level` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `use_cudagraph` is not supported in vLLM.conf

<frozen importlib._bootstrap_external>:1329: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1329: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


INFO 11-13 18:13:12 [core.py:105] Initializing a V1 LLM engine (v0.19.1) with config: model='/root/.cache/huggingface/hub/models--unsloth--Qwen3-1.7B/snapshots/6262b50d6c1f8ee5e4ac750d710c33603bfc2a0c', speculative_config=None, tokenizer='/root/.cache/huggingface/hub/models--unsloth--Qwen3-1.7B/snapshots/6262b50d6c1f8ee5e4ac750d710c33603bfc2a0c', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', e

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 11-13 18:13:16 [default_loader.py:384] Loading weights took 1.09 seconds
INFO 11-13 18:13:16 [punica_selector.py:20] Using PunicaWrapperGPU.
INFO 11-13 18:13:17 [gpu_model_runner.py:4820] Model loading took 3.31 GiB memory and 1.511822 seconds
INFO 11-13 18:13:30 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/cb93a9228d/rank_0_0/backbone for vLLM's torch.compile
INFO 11-13 18:13:30 [backends.py:1111] Dynamo bytecode transform time: 12.20 s


Unsloth: Compiling kernels: 100%|██████████| 7/7 [00:00<00:00,  7.97it/s, triton_poi_fused__to_copy_add_index_select_mean_mul_pow_rsqrt_split_split_with_sizes_sub_unsqueeze_view_6]

... (9 outputs left out)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['k_norm', 'attention_norm', 'q_layernorm', 'layer_norm1', 'input_layernorm', 'k_layernorm', 'pre_feedforward_layernorm_2', 'post_feedforward_layernorm_1', 'post_layernorm', 'operator_norm', 'q_norm', 'norm2', 'cross_attn_input_layernorm', 'post_per_layer_input_norm', 'post_feedforward_layernorm_2', 'post_attention_layernorm', 'norm1', 'post_feedforward_layernorm', 'pre_feedforward_layernorm', 'norm', 'cross_attn_post_attention_layernorm', 'ffn_norm', 'layer_norm2']


Unsloth 2026.11.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Loaded 392 LoRA tensors from /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/sft_lora
[cache] bench_sft: reusing saved result
| metric | base | sft |
|---|---|---|
| good rate | 62.5% | 72.9% |
| relevance (1-4) | 3.33 | 3.73 |
| factuality (1-4) | 2.75 | 3.31 |
| tool cost / query | 0.82 | 0.82 |
| excess cost / query | 0.16 | 0.04 |
| knowledge_search calls / query | 0.62 | 0.54 |
| tool rounds / query | 1.12 | 1.29 |
| invalid call rate | 4.4% | 1.2% |
| final answer rate | 100.0% | 97.9% |
| parallel rate on fan-out tasks | 50.0% | 88.9% |

Good rate by task family:

| family | base | sft |
|---|---|---|
| kb_bridge | 50.0% | 66.7% |
| kb_compare | 50.0% | 66.7% |
| weather_lookup | 100.0% | 100.0% |
| election_lookup | 83.3% | 100.0% |
| fanout_calc | 33.3% | 83.3% |
| tool_bridge | 50.0% | 100.0% |
| composite | 33.3% | 0.0% |
| no_tool | 100.0% | 66.7% |

Failure tags (share of all tasks):

| tag | base | sft |
|---|---|---|
| specialized_tool_not_called | 10.4% | 

## 5. The environment, the tools and the reward

The four tools are plain Python functions with type hints and docstrings; TRL derives their schemas and runs them
inside the rollout loop. `knowledge_search` is BM25 over a local Wikipedia-paragraph index and stands in for the
expensive retrieval tool (cost 1.0 per call; the others cost 0.25).

The reward needs no LLM judge. Every training task carries gold labels: the answer, and the evidence a correct plan
must retrieve. Evidence is read only from tool messages, which Python writes, so the model cannot fake it.

```
g = (min(relevance, factuality) - 1) / 3                       # 0, 1/3, 2/3 or 1
R = g - 0.3 * min(1, excess_cost / 3) - 0.2 * invalid_fraction   # "graded_cost"
```

`excess_cost` is the cost above the task's cheapest correct plan: necessary calls are free, redundant ones are not.
Taking the minimum of the two axes means guessing an answer without evidence earns at most 1/3, and retrieving
evidence without answering earns 0.

In [8]:
from planner.rewards import ToolLedger, make_reward_fn
from planner.scoring import score_messages
from planner.tasks import grpo_rows
from planner.tools import MAX_TOOL_ROUNDS, TOOLS, execute_tool

task = next(t for t in env.tasks["grpo"] if t["family"] == "tool_bridge")
print("Q:", task["query"])
for calls in task["optimal_plan"]:
    for c in calls: print(" ", c["tool"], c["args"], "->", execute_tool(c["tool"], c["args"])[:160])
print("gold answer:", task["aspects"][0]["answers"], "| cheapest plan costs", task["optimal_cost"])

Q: How much precipitation fell in the capital of Velland on the day of its 2016 election?
  get_election {'region': 'Velland', 'year': 2016} -> {"region": "Velland", "year": 2016, "winner": "Emeric Kessler", "winner_party": "Progress League", "winner_vote_share_pct": 51.5, "runner_up": "Joss Zeller", "r
  get_weather {'city': 'Dorburg', 'date': '2016-11-09'} -> {"city": "Dorburg", "date": "2016-11-09", "temp_c": 2.2, "condition": "foggy", "precip_mm": 0.0}
gold answer: ['0.0'] | cheapest plan costs 0.5


## 6. GRPO

Each optimizer step uses `grpo_batch * grpo_accum / grpo_num_generations` prompts with `grpo_num_generations` rollouts
each. Rollouts stop when the model answers without a tool call, after three tool rounds, or when
`max_completion_length` (which counts tool results too) is used up. No KL term (`beta = 0`).

Logged to W&B every step: the reward, good rate, both axes, tool cost, the share of groups whose rollouts all got the
same reward (those carry no learning signal), TRL's `tools/call_frequency` and `tools/failure_frequency`, and a running
count of calls per tool under `qps/` - an estimate of what this run would cost against real tools.

In [9]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer
from planner.persist import latest_checkpoint, log_summary, wandb_init
from planner.unsloth_utils import filter_kwargs, save_adapter

os.makedirs(GRPO_DIR, exist_ok=True)
wandb_run, REPORT_TO = wandb_init(GRPO_DIR, WANDB_PROJECT, f"{STAGE}-{cfg.repo_stem}", cfg.to_dict())
ledger = ToolLedger(os.path.join(GRPO_DIR, "tool_ledger.json"))

if RESUME and os.path.exists(os.path.join(GRPO_LORA, "adapter_model.safetensors")):
    load_lora_weights(model, GRPO_LORA)
    print("GRPO already finished: adapter loaded from Drive")
else:
    FastLanguageModel.for_training(model)
    dataset = Dataset.from_list(grpo_rows(env.tasks["grpo"], cfg.seed))
    extra = {"chat_template_kwargs": {"enable_thinking": cfg.enable_thinking}} if fmt.is_thinking_model else {}
    args = GRPOConfig(**filter_kwargs(GRPOConfig, dict(
        temperature = 1.0,
        learning_rate = cfg.grpo_lr,
        weight_decay = 0.001,
        warmup_ratio = 0.1,
        lr_scheduler_type = "linear",
        optim = "adamw_8bit",
        beta = 0.0,
        logging_steps = 1,
        per_device_train_batch_size = cfg.grpo_batch,
        gradient_accumulation_steps = cfg.grpo_accum,
        num_generations = cfg.grpo_num_generations,
        max_completion_length = cfg.grpo_max_completion,
        max_tool_calling_iterations = MAX_TOOL_ROUNDS,
        max_steps = cfg.grpo_steps,
        save_steps = cfg.grpo_save_steps,
        save_total_limit = 2,
        seed = cfg.seed,
        report_to = REPORT_TO,
        run_name = f"{STAGE}-{cfg.repo_stem}",
        output_dir = GRPO_DIR,
        **extra,
    )))
    trainer = GRPOTrainer(
        model = model,
        processing_class = tokenizer,
        reward_funcs = [make_reward_fn(cfg.reward_mode, ledger = ledger, print_every = 20)],
        tools = TOOLS,
        args = args,
        train_dataset = dataset,
    )
    checkpoint = latest_checkpoint(GRPO_DIR) if RESUME else None
    print("resume from checkpoint:", checkpoint)
    trainer.train(resume_from_checkpoint = checkpoint)
    save_adapter(model, tokenizer, fmt, GRPO_LORA)
print("tool calls made during RL:", dict(ledger.calls), "| rollouts:", ledger.rollouts, "| total cost:", round(ledger.total_cost, 1))

W&B disabled (set WANDB_API_KEY and a project to enable it)
GRPOConfig does not accept ['warmup_ratio']: skipped
resume from checkpoint: None


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 128 | Num Epochs = 1 | Total steps = 4
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 34,865,152 of 1,755,440,128 (1.99% trained)


Unsloth: Merged-weight rollouts on (all, automatic): folding the adapter into 196 shared weights; exact fold / unfold, no pristine copy (only non-round-tripping elements are kept, in pinned host memory).
Unsloth: exact merged rollout restores 3,264,214 of 1,409,286,144 weights (0.23%) from a 19.3 MB host store.
******************** 
[election_lookup] Forget the forecast for Caldbridge for a moment. Which candidate won in Zanland in 2020?
[assistant] [{'type': 'function', 'function': {'name': 'get_election', 'arguments': {'region': 'Zanland', 'year': 2020}}}]
[tool] {"region": "Zanland", "year": 2020, "winner": "Tessa Wexford", "winner_party": "Heritage Union", "winner_vote_share_pct": 50.0, "runner_up": "Lev Thackeray", "runner_up_party": "Civic Front", "runner_up_vote_share_pct": 30.9, "turnout_pct": 50.5, "election_date": "2020-05-20", "capital": "Caldbridge"}
[assistant] Answer: Tessa Wexford
-> relevance=4 factuality=4 cost=0.25 reward=1.000 tags=[]
Unsloth: Will smartly offload gr

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,tools / call_frequency,tools / failure_frequency,kl,rewards / reward_graded_cost / mean,rewards / reward_graded_cost / std
1,0.000000,1.000000,0.000000,40.000000,40.000000,40.000000,0.000000,40.000000,40.000000,40.000000,1.000000,0.000000,0.000000,1.000000,0.000000
2,0.000000,0.000000,0.000000,37.000000,30.000000,58.000000,0.000000,37.000000,30.000000,58.000000,1.250000,0.000000,0.000000,0.000000,0.000000
3,0.000000,1.000000,0.000000,39.000000,39.000000,39.000000,0.000000,39.000000,39.000000,39.000000,1.000000,0.000000,0.000000,1.000000,0.000000
4,-0.029025,0.814583,0.216279,102.750000,80.000000,115.000000,0.000000,102.750000,80.000000,115.000000,2.750000,0.000000,0.000000,0.814583,0.216279


DEBUG:bm25s:Building index from IDs objects


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/grpo_graded_cost/checkpoint-2/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/grpo_graded_cost/checkpoint-4/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/tool_use_demo/planner_qwen3-1.7b_smoke/grpo_graded_cost_lora/tokenizer_config.json.


tool calls made during RL: {'get_election': 8, 'knowledge_search': 6, 'get_weather': 8, 'calculator': 2} | rollouts: 16 | total cost: 10.5


## 7. Evaluate the GRPO model

In [10]:
BENCH["grpo"] = bench(STAGE, GRPO_LORA)
print(format_report(BENCH))
for name, report in BENCH.items(): log_summary(wandb_run, f"bench_{name}", report["summary"])

Rendering prompts:   0%|          | 0/16 [00:00<?, ?it/s]

WARNING 11-13 18:17:26 [input_processor.py:149] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/16 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

... (12 outputs left out)

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  bench: 48/48
| metric | base | sft | grpo |
|---|---|---|---|
| good rate | 62.5% | 72.9% | 72.9% |
| relevance (1-4) | 3.33 | 3.73 | 3.77 |
| factuality (1-4) | 2.75 | 3.31 | 3.33 |
| tool cost / query | 0.82 | 0.82 | 0.85 |
| excess cost / query | 0.16 | 0.04 | 0.04 |
| knowledge_search calls / query | 0.62 | 0.54 | 0.58 |
| tool rounds / query | 1.12 | 1.29 | 1.25 |
| invalid call rate | 4.4% | 1.2% | 1.2% |
| final answer rate | 100.0% | 97.9% | 100.0% |
| parallel rate on fan-out tasks | 50.0% | 88.9% | 94.4% |

Good rate by task family:

| family | base | sft | grpo |
|---|---|---|---|
| kb_bridge | 50.0% | 66.7% | 66.7% |
| kb_compare | 50.0% | 66.7% | 66.7% |
| weather_lookup | 100.0% | 100.0% | 100.0% |
| election_lookup | 83.3% | 100.0% | 100.0% |
| fanout_calc | 33.3% | 83.3% | 83.3% |
| tool_bridge | 50.0% | 100.0% | 100.0% |
| composite | 33.3% | 0.0% | 0.0% |
| no_tool | 100.0% | 66.7% | 66.7% |

Failure tags (share of all tasks):

| tag | base | sft | grpo |
|---|---|-

## 8. Save and push

In [ ]:
from planner.persist import model_card
from planner.unsloth_utils import export_model

card = model_card(
    f"{cfg.repo_stem}-{HUB_STAGE}",
    cfg.base_model_hf,
    f"Tool-calling planner after SFT and GRPO (reward: `{cfg.reward_mode}`). It plans with `knowledge_search`, `calculator`, "
    "`get_weather` and `get_election` for at most three tool rounds and ends with an `Answer:` line. Code: " + REPO_URL,
    {"PlannerQA-Bench (held-out)": format_report(BENCH), "BFCL": bfcl.format_table({k: v for k, v in BFCL.items() if v}) if any(BFCL.values()) else "not run"},
)
status = export_model(model, tokenizer, fmt, cfg, HUB_STAGE, HF_USER, GRPO_LORA, GRPO_MERGED, card, push = PUSH_TO_HUB, private = HF_PRIVATE, gguf = PUSH_GGUF)
print(status)

Unsloth: Restored added_tokens_decoder metadata in /content/work/planner_qwen3-1.7b_smoke/grpo_graded_cost_merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/content/work/planner_qwen3-1.7b_smoke/grpo_graded_cost_merged`: 100%|██████████| 1/1 [00:00<00:00, 1964.55it/s]


Successfully copied all 1 files from cache to `/content/work/planner_qwen3-1.7b_smoke/grpo_graded_cost_merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:12<00:00, 12.26s/it]


Unsloth: Merge process complete. Saved to `/content/work/planner_qwen3-1.7b_smoke/grpo_graded_cost_merged`
[hub] pushed grpo adapter
[hub] pushed grpo merged model
Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


... (2 outputs left out)

Unsloth: Copying 1 files from cache to `/tmp/unsloth_gguf_q493tzm0`: 100%|██████████| 1/1 [00:00<00:00, 2695.57it/s]


Successfully copied all 1 files from cache to `/tmp/unsloth_gguf_q493tzm0`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:11<00:00, 11.99s/it]


Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_q493tzm0`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q8_0', 'q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b12340-mix-6186f42 (app-b12340-mix-6186f42-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...


## 9. BFCL on the GRPO checkpoint, and the final comparison

BFCL needs most of the GPU for its own vLLM server. The cell first tries to release the training model. If the GPU is
still busy it says so: then choose *Runtime > Restart session* and *Run all*. Every finished stage is skipped, and the
BFCL cell in section 3 scores the GRPO checkpoint.

In [ ]:
from planner.unsloth_utils import gpu_free_fraction, release_gpu

if RUN_BFCL and not BFCL.get("grpo"):
    try: del trainer
    except NameError: pass
    del model
    free = release_gpu()
    if free >= 0.7:
        BFCL["grpo"] = bfcl.run_stage(CACHE, STAGE, cfg, PERSIST, local_model_path=GRPO_MERGED, enabled=True, gpu_memory_utilization=round(0.9 * free, 2))
    else:
        print(f"Only {free:.0%} of the GPU is free. Restart the session and Run all: section 3 will score the GRPO checkpoint.")

report = "## PlannerQA-Bench (800 held-out tasks)\n\n" + format_report(BENCH)
shown = {k: v for k, v in BFCL.items() if v}
if shown:
    report += "\n\n## BFCL (official checker, offline categories)\n\n" + bfcl.format_table(shown)
    for name, result in shown.items(): log_summary(wandb_run, f"bfcl_{name}", dict(result["groups"], bfcl_core=result["bfcl_core"]))
report += f"\n\n## Tool calls made during RL\n\n{dict(ledger.calls)} over {ledger.rollouts} rollouts (cost {ledger.total_cost:.1f})\n"
with open(os.path.join(PERSIST, f"report_{STAGE}.md"), "w") as f: f.write(report)
if wandb_run: wandb_run.finish()

from IPython.display import Markdown, display
display(Markdown(report))

### Reading the result

- **SFT** is expected to move BFCL and the simpler task families: it teaches formats, schemas and the tool inventory
  with dense per-token supervision.
- **GRPO** is expected to move the hard families (`kb_bridge`, `tool_bridge`, `composite`) and the tool cost, because it
  trains on the states the model itself reaches and optimises the outcome directly. On BFCL it should hold steady: the
  GRPO environment has a fixed tool set, so BFCL here is a check that general function calling was not lost.
- The failure-tag table shows which error types each stage removed.

These are expectations, not guarantees: compare them with the tables above. To try a different reward, change
`REWARD_MODE` and run again - SFT results and the environment are reused.